# BETWEEN THE NOTES : Autonomous Analog Horror Cinema Engine
### A 5-Minute (300s) Psychological Analog Horror Film set to "Noir" by Atandra
**Presented by Between The Notes**
**Visual Engine & Direction: Sayan | Score & Sound Design: Atandra**
**Sub-Theme: নিশিডাক (THE NISHIDAK BROADCAST) : Rural Bengal Analog Horror**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayan-bhattacharya/noir-analog-horror/blob/main/colab_noir_accelerator.ipynb)

This engine renders a complete 5-minute Indian / Bengali analog horror film featuring:
- **Between the Notes Network & Truman-Show Channel Switching**: `CH 03` (News / Hyper-Objects) vs `CH 07` (Live Surveillance / Descent Cam).
- **Act V: DESCENT...**: The protagonist wakes to the Nishir Daak, falls into a viscous black gluey meta-object hole, and rotates endlessly through the void.
- **Locked-Off Sky Time-Lapse**: Photorealistic 24fps nocturnal cloud time-lapse composited behind stationary village huts, fences, and trees.
- **HarfBuzz + FreeType Bengali OpenType Shaping**: 100% authentic conjuncts and zero tofu glyphs.
- **Master Audio Sync**: Synchronized to Between the Notes audio (`BetweenTheNotes_Podcast_5min.wav` / `Noir.wav`).
- **Automated AI Watermark Scrubber**: 100% removal of Gemini/SynthID corner watermarks.


## 1. Hardware Detection & Verification


In [ ]:
import torch
if torch.cuda.is_available():
    print(f"[SUCCESS] GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[INFO] Running on Standard CPU. For fast renders, select Runtime -> Change runtime type -> T4 GPU")

## 2. Instant Clone of Public NOIR Engine & Dependency Setup


In [ ]:
import os
import sys

%cd /content
!rm -rf noir-analog-horror
!git clone https://github.com/sayan-bhattacharya/noir-analog-horror.git /content/noir-analog-horror

project_dir = "/content/noir-analog-horror"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
os.environ["PYTHONPATH"] = project_dir
%cd /content/noir-analog-horror

!pip install -q opencv-python soundfile scipy pillow uharfbuzz freetype-py pyyaml

# Auto-patch OpenCV 5 putText CV_8U compatibility
frame_py = '/content/noir-analog-horror/noir_film/frame.py'
if os.path.exists(frame_py):
    with open(frame_py, 'r') as f: src = f.read()
    src = src.replace('dtype=np.float32', 'dtype=np.uint8')
    src = src.replace('.astype(np.float32)', '.astype(np.uint8)')
    src = src.replace('np.zeros((height, width, 3), np.float32)', 'np.zeros((height, width, 3), np.uint8)')
    with open(frame_py, 'w') as f: f.write(src)

shaders_py = '/content/noir-analog-horror/noir_film/shaders.py'
if os.path.exists(shaders_py):
    with open(shaders_py, 'r') as f: src = f.read()
    src = src.replace('curved = cv2.remap(frame, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))', 'curved = cv2.remap(frame, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0)).astype(np.float32)')
    src = src.replace('return np.clip(curved, 0, 255).astype(np.float32)', 'return np.clip(curved, 0, 255).astype(np.uint8)')
    src = src.replace('return np.clip(frame + noise * (amount / 0.08), 0, 255)', 'return np.clip(frame.astype(np.float32) + noise * (amount / 0.08), 0, 255).astype(np.uint8)')
    src = src.replace('return out', 'return out.astype(np.uint8)')
    with open(shaders_py, 'w') as f: f.write(src)

render_py = '/content/noir-analog-horror/noir_film/render_noir.py'
if os.path.exists(render_py):
    with open(render_py, 'r') as f: src = f.read()
    src = src.replace("'-ss', str(start_offset)", "'-stream_loop', '-1', '-ss', str(start_offset)")
    with open(render_py, 'w') as f: f.write(src)

# Ensure Noir.wav audio source is ready
noir_wav = '/content/noir-analog-horror/Noir.wav'
preview_mp4 = '/content/noir-analog-horror/noir_720p_story_preview.mp4'
if not os.path.exists(noir_wav) and os.path.exists(preview_mp4):
    print('Extracting original soundtrack from preview...')
    !ffmpeg -y -i /content/noir-analog-horror/noir_720p_story_preview.mp4 -vn -c:a pcm_s16le /content/noir-analog-horror/Noir.wav

print("\n[READY] NOIR Engine initialized in:", os.getcwd())

## 3. High-Performance 5-Minute Master Render Execution
Renders the complete 300-second (7,200 frames @ 24 fps) Bengali analog horror film with calibrated reading times and pure *Noir.wav* audio synchronization.


In [ ]:
import os
import time

%cd /content/noir-analog-horror
os.environ["PYTHONPATH"] = "."

print("Starting 5-Minute Master Film Render (300 seconds, 7,200 frames)...\n")
start_time = time.time()

# Render full 6-minute (360s) first act master (quality: final = 720p HD CRF 16; use --preview for fast draft)
!python -m noir_film.render_noir --duration 360 --quality final

elapsed = time.time() - start_time
print(f"\n[FINISHED] Full 300s master rendered in {elapsed:.1f}s")

## 4. In-Notebook Video Player


In [ ]:
from IPython.display import HTML
from base64 import b64encode
import glob
import os

# Locate newly rendered master MP4 from output/
mp4s = sorted(glob.glob("/content/noir-analog-horror/output/*.mp4"), key=os.path.getmtime, reverse=True)
video_path = mp4s[0] if mp4s else "/content/noir-analog-horror/noir_master_output.mp4"

if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / (1024 * 1024)
    print(f"[READY] Playing master film: {os.path.basename(video_path)} ({size_mb:.2f} MB)")
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=720 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("Video file not found at:", video_path)

## 5. Download Master Film (720p HD MP4)


In [ ]:
from google.colab import files
import glob
import os

mp4s = sorted(glob.glob("/content/noir-analog-horror/output/*.mp4"), key=os.path.getmtime, reverse=True)
target = mp4s[0] if mp4s else "/content/noir-analog-horror/noir_master_output.mp4"

if os.path.exists(target):
    size_mb = os.path.getsize(target) / (1024 * 1024)
    print(f"Downloading master film: {os.path.basename(target)} ({size_mb:.2f} MB)...\n")
    files.download(target)
else:
    print("Target video not found at:", target)

## 6. Save Master to Google Drive (Remote Backup & Streaming)


In [ ]:
import os
import shutil
import glob
from google.colab import drive

print("Mounting Google Drive...")
drive.mount("/content/drive")

drive_folder = "/content/drive/MyDrive/NOIR_Film"
os.makedirs(drive_folder, exist_ok=True)

mp4s = sorted(glob.glob("/content/noir-analog-horror/output/*.mp4"), key=os.path.getmtime, reverse=True)
src = mp4s[0] if mp4s else "/content/noir-analog-horror/noir_master_output.mp4"

if os.path.exists(src):
    dest = os.path.join(drive_folder, os.path.basename(src))
    shutil.copy2(src, dest)
    print(f"[SUCCESS] Saved 5-minute master to Google Drive: {dest}")
    if hasattr(os, "sync"):
        os.sync()
    print("\n[SYNC COMPLETE] Ready to stream on your phone/TV!")
else:
    print("Source video not found at:", src)